In [49]:
from pathlib import Path
import json

import numpy as np
import pandas as pd


RESULTS_FOLDER = Path(
    "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/benchmark/downloads/results/bio/base_run"
)

In [50]:
records = []

for config_path in sorted(RESULTS_FOLDER.glob("*/seed_*/config.json")):
    config = json.loads(config_path.read_text())
    metrics = json.loads(config_path.with_name("metrics.json").read_text())

    model_type = config["predictor_config"]["type"]
    if config["rearrangement_config"] is not None:
        model_type += "_rearranged"

    calibrator = config["conformal_config"]["calibrator"]
    conformal_score_type = calibrator["type"]
    if conformal_score_type == "norm":
        conformal_score_type = f"l{calibrator['p']:g}"

    for metric_name, metric in metrics.items():
        if isinstance(metric, dict):
            records.append(
                {
                    "model_type": model_type,
                    "conformal_score_type": conformal_score_type,
                    "seed": config["seed"],
                    "metric": metric_name,
                    "seed_mean": metric["mean"],
                }
            )

results = pd.DataFrame(records)
results = results.loc[
    (results["metric"] != "log_volume_per_dimension")
    | np.isfinite(results["seed_mean"])
]
statistics = (
    results.groupby(["model_type", "conformal_score_type", "metric"])["seed_mean"]
    .agg(mean="mean", std="std")
)

In [51]:
for (model_type, conformal_score_type), table in statistics.groupby(
    level=["model_type", "conformal_score_type"]
):
    print("=" * 80)
    print(f"Model: {model_type}")
    print(f"Conformal score: {conformal_score_type}\n")
    print(
        table.droplevel(["model_type", "conformal_score_type"])
        .rename(
            columns={
                "mean": "Mean across seeds",
                "std": "Std of seed means",
            }
        )
        .to_string(float_format=lambda value: f"{value:.6f}")
    )
    print()

Model: neural_optimal_transport
Conformal score: l2

                          Mean across seeds  Std of seed means
metric                                                        
excess_coverage_risk               0.063124           0.002923
log_volume_per_dimension           4.575632           0.006532
marginal_coverage                  0.901159           0.001748
worst_slab_coverage                0.850012           0.047878

Model: neural_optimal_transport_rearranged
Conformal score: l2

                          Mean across seeds  Std of seed means
metric                                                        
excess_coverage_risk               0.063291           0.001483
log_volume_per_dimension           4.540579           0.012065
marginal_coverage                  0.902318           0.001192
worst_slab_coverage                0.846418           0.035974

Model: normalizing_flow
Conformal score: cdf_calibrator

                          Mean across seeds  Std of seed means
metri

In [58]:
# Add or remove result roots and metric names here.
RESULTS_FOLDERS = [
    Path(
        "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/"
        "benchmark/downloads/results/bio/base_run"
    ),
    Path(
        "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/"
        "benchmark/downloads/results/blog/base_run"
    ),
    Path(
        "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/"
        "benchmark/downloads/results/sgemm/base_run"
    ),
    Path(
        "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/"
        "benchmark/downloads/results/qm9/base_run"
    ),
    Path(
        "/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/"
        "benchmark/downloads/results/scm20d"
    ),
]
SELECTED_METRICS = [
    "log_volume_per_dimension",
    "worst_slab_coverage",
]
# Case-insensitive substrings matched against experiment and model names.
# Set to None to include every model.
SELECTED_MODELS = ["rearranged", "transport_realnvp_l2", "transport_neural_ot"]


def print_selected_metrics(
    results_folders, selected_metrics, selected_models=None
):
    """Print selected metric means and cross-seed standard deviations."""
    summaries = {}
    model_patterns = tuple(
        str(model).casefold() for model in (selected_models or ())
    )

    for results_folder in map(Path, results_folders):
        print("=" * 120)
        print(results_folder)

        if not results_folder.is_dir():
            print("Folder does not exist.\n")
            continue

        config_paths = sorted(
            results_folder.glob("*/seed_*/config.json")
        )
        if not config_paths:
            print("No */seed_*/config.json files found.\n")
            continue

        records = []
        skipped_metrics_files = []
        for config_path in config_paths:
            metrics_path = config_path.with_name("metrics.json")
            if not metrics_path.is_file():
                skipped_metrics_files.append(metrics_path)
                continue

            config = json.loads(config_path.read_text(encoding="utf-8"))
            metrics = json.loads(metrics_path.read_text(encoding="utf-8"))

            model_type = config.get("predictor_config", {}).get(
                "type", "unknown"
            )
            if config.get("rearrangement_config") is not None:
                model_type += "_rearranged"

            conformal_config = config.get("conformal_config") or {}
            calibrator = conformal_config.get("calibrator") or {}
            score_type = calibrator.get("type", "unknown")
            if score_type == "norm":
                score_type = f"l{calibrator['p']:g}"

            experiment_name = config_path.parent.parent.name
            model_names = (experiment_name.casefold(), model_type.casefold())
            if model_patterns and not any(
                pattern in name
                for pattern in model_patterns
                for name in model_names
            ):
                continue

            record = {
                # "experiment": experiment_name,
                "model_type": model_type,
                # "conformal_score_type": score_type,
                # "seed": config.get("seed", config_path.parent.name),
            }
            for metric_name in selected_metrics:
                metric = metrics.get(metric_name)
                if isinstance(metric, dict):
                    metric = metric.get("mean")
                record[metric_name] = (
                    float(metric)
                    if isinstance(metric, (int, float))
                    else np.nan
                )
            records.append(record)

        if skipped_metrics_files:
            print(
                f"Skipped {len(skipped_metrics_files)} runs without "
                "metrics.json."
            )
        if not records:
            print("No readable result records found.\n")
            continue

        seed_metrics = pd.DataFrame(records)
        seed_metrics[list(selected_metrics)] = seed_metrics[
            list(selected_metrics)
        ].replace([np.inf, -np.inf], np.nan)
        group_columns = [
            # "experiment",
            "model_type",
            # "conformal_score_type",
        ]
        grouped = seed_metrics.groupby(group_columns, sort=True)
        metric_statistics = grouped[list(selected_metrics)].agg(
            ["mean", "std"]
        )
        metric_statistics.columns = [
            f"{metric} ({statistic})"
            for metric, statistic in metric_statistics.columns
        ]
        summary = pd.concat(
            [
                # grouped["seed"].nunique().rename("n_seeds"),
                metric_statistics,
            ],
            axis=1,
        ).reset_index()

        summaries[str(results_folder)] = summary
        print(
            summary.to_string(
                index=False,
                na_rep="--",
                float_format=lambda value: f"{value:.6f}",
            )
        )
        print()

    return summaries


selected_metric_summaries = print_selected_metrics(
    RESULTS_FOLDERS, SELECTED_METRICS, SELECTED_MODELS
)

/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/benchmark/downloads/results/bio/base_run
                         model_type  log_volume_per_dimension (mean)  log_volume_per_dimension (std)  worst_slab_coverage (mean)  worst_slab_coverage (std)
           neural_optimal_transport                         4.575632                        0.006532                    0.850012                   0.047878
neural_optimal_transport_rearranged                         4.540579                        0.012065                    0.846418                   0.035974
                   normalizing_flow                         4.601297                        0.083386                    0.882692                   0.016591
        normalizing_flow_rearranged                         4.476399                        0.017632                    0.846994                   0.016159

/Users/vladimir.kondratyev/minimal_volume_conformal_prediction/benchmark/downloads/results/blog/base_run
          

In [ ]:
bio
                         model_type  log_volume_per_dimension (mean)  log_volume_per_dimension (std)  worst_slab_coverage (mean)  worst_slab_coverage (std)
           neural_optimal_transport                         4.575632                        0.006532                    0.850012                   0.047878
neural_optimal_transport_rearranged                         4.540579                        0.012065                    0.846418                   0.035974
                   normalizing_flow                         4.601297                        0.083386                    0.882692                   0.016591
        normalizing_flow_rearranged                         4.476399                        0.017632                    0.846994                   0.016159

blog
                         model_type  log_volume_per_dimension (mean)  log_volume_per_dimension (std)  worst_slab_coverage (mean)  worst_slab_coverage (std)
           neural_optimal_transport                         2.979691                        0.087840                    0.815420                   0.022608
neural_optimal_transport_rearranged                         2.902963                        0.108433                    0.834413                   0.016926
                   normalizing_flow                         2.855625                        0.033835                    0.821305                   0.009790
        normalizing_flow_rearranged                         2.819970                        0.031209                    0.828808                   0.022495

sgemm
                         model_type  log_volume_per_dimension (mean)  log_volume_per_dimension (std)  worst_slab_coverage (mean)  worst_slab_coverage (std)
           neural_optimal_transport                        -2.062205                        0.040081                    0.803587                   0.014276
neural_optimal_transport_rearranged                        -2.065758                        0.041675                    0.806713                   0.023144
                   normalizing_flow                        -1.777873                        0.048733                    0.813585                   0.012284
        normalizing_flow_rearranged                        -1.802323                        0.063916                    0.811707                   0.018846

qm9
                         model_type  log_volume_per_dimension (mean)  log_volume_per_dimension (std)  worst_slab_coverage (mean)  worst_slab_coverage (std)
           neural_optimal_transport                         0.440911                        0.010858                    0.870936                   0.008619
neural_optimal_transport_rearranged                         0.437382                        0.010180                    0.873090                   0.007799
                   normalizing_flow                         0.196083                        0.020905                    0.888268                   0.014345
        normalizing_flow_rearranged                         0.194550                        0.021956                    0.882928                   0.006317

scm20d
                         model_type  log_volume_per_dimension (mean)  log_volume_per_dimension (std)  worst_slab_coverage (mean)  worst_slab_coverage (std)
           neural_optimal_transport                         6.312482                        0.029469                    0.841514                   0.026869
neural_optimal_transport_rearranged                         6.301987                        0.022155                    0.845205                   0.050286
                   normalizing_flow                         7.164626                        0.166723                    0.879879                   0.034240
        normalizing_flow_rearranged                         6.762695                        0.067801                    0.865088                   0.028312